# 1 — Clustering within a cohort

Hierarchical clustering and k-means, **each with AU *p*-values**, run separately
inside each cohort. This is day one: every cohort works alone.

The question this notebook answers is not *what are the clusters* — any method will
hand you clusters. It is **which of them are real**.

| | gives you clusters | tells you if they're real |
|---|---|---|
| k-means | ✅ (you pick k) | ❌ |
| hclust | ✅ (you pick the cut) | ❌ |
| **pvclust** | ✅ | ✅ AU *p*-value per cluster |

In [ ]:
import warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram

from pvclust_py.core import pvclust, kmeans_pv, pvpick
from pvclust_py.datasets import load_cohorts

pd.set_option('display.width', 120)

# Set the clustering configuration ONCE, here.
# minkowski + ward.D2 is the pairing carried over from microarray/RNAseq work.
# In pvclust, minkowski means p=2 -- i.e. euclidean -- because dist.pvclust never
# forwards p to R's dist(). Verified against R: max difference 0.0. Conveniently,
# that also means it pools exactly under federation.
METHOD_DIST = 'minkowski'
METHOD_HCLUST = 'ward.D2'

## Your data goes here

`load_cohorts()` splits a demo matrix by rows. **Swap in your own** — the only
requirement is the orientation:

- **rows = resampling units** (samples / patients)
- **columns = the objects being clustered** (proteins for SomaScan, genes for RNAseq)

If your matrix is the other way round, pass `df.T`. This matters: the bootstrap
resamples rows, so rows must be things you could have drawn more of.

In [ ]:
cohorts = load_cohorts(n_cohorts=3)

# --- to use your own data instead, replace the above with something like: -----
# soma = pd.read_csv('somascan.csv', index_col=0)   # rows=samples, cols=proteins
# cohorts = {'cohort_A': soma[soma.arm == 'A'].drop(columns='arm'), ...}

for name, df in cohorts.items():
    print(f'{name}: {df.shape[0]} samples x {df.shape[1]} objects to cluster')

## Hierarchical clustering with AU

`nboot=1000` is R's default. Lower it while exploring — it is the whole cost.

In [ ]:
NBOOT = 200   # raise to 1000 (R's default) for real results

results = {}
for name, df in cohorts.items():
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        results[name] = pvclust(df.to_numpy(float), list(df.columns),
                                method_dist=METHOD_DIST, method_hclust=METHOD_HCLUST,
                                nboot=NBOOT, seed=42)
    print(f'{name}: {len(results[name].edges)} clusters evaluated')

### AU vs BP — why this is not just a bootstrapped tree

`BP` is the ordinary bootstrap number: *how often did this cluster reappear?*
It is **biased downward**. `AU` removes that bias.

Watch the gap in the table below. Anything where AU is high and BP is low is a
real cluster that a plain bootstrap would have told you to throw away.

In [ ]:
r = results['cohort_A']
tab = r.edges_frame()[['n_members', 'bp', 'au', 'si', 'se_au', 'pchi']]
tab = tab.sort_values('au', ascending=False)
tab['au_minus_bp'] = tab['au'] - tab['bp']
tab.head(10)

**`pchi` is the honesty check.** It is a goodness-of-fit for the curve AU is read
off. A small `pchi` means the model did not describe the data and the AU on that
row should not be trusted, however attractive it looks.

## Choosing clusters without choosing k

`pvpick` returns the significant clusters. You supply a *confidence* threshold,
not a number of clusters — the tree cuts itself.

In [ ]:
for name, res in results.items():
    picked = pvpick(res, alpha=0.95, use='au')
    print(f'\n{name}: {len(picked)} clusters at AU >= 0.95')
    for e in picked:
        print(f"   AU={e['au']:.3f}  BP={e['bp']:.3f}  n={e['n_members']:2d}  "
              f"{', '.join(e['members'][:3])}{'...' if e['n_members'] > 3 else ''}")

### The effect of your choices

Every knob moves the answer. Look before you settle on one.

In [ ]:
df = cohorts['cohort_A']
rows = []
for dist in ['minkowski', 'correlation', 'abscor']:
    for link in ['ward.D2', 'average']:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            res = pvclust(df.to_numpy(float), list(df.columns), method_dist=dist,
                          method_hclust=link, nboot=200, seed=42)
        rows.append({'distance': dist, 'linkage': link,
                     'clusters at AU>=0.95': len(pvpick(res, 0.95)),
                     'max AU': max(e['au'] for e in res.edges[:-1])})
pd.DataFrame(rows)

In [ ]:
# nboot buys precision, and costs time. se_au shows what you are buying.
rows = []
for nb in [100, 250, 500, 1000]:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = pvclust(df.to_numpy(float), list(df.columns), nboot=nb, seed=42)
    fitted = [e for e in res.edges if e['df'] > 0]
    rows.append({'nboot': nb, 'median se_au': np.median([e['se_au'] for e in fitted]),
                 'clusters at AU>=0.95': len(pvpick(res, 0.95))})
pd.DataFrame(rows)

## The dendrogram, labelled with AU

In [ ]:
res = results['cohort_A']
fig, ax = plt.subplots(figsize=(11, 5))
dd = dendrogram(res.linkage, labels=res.labels, ax=ax, color_threshold=0,
                above_threshold_color='#555555')
for e in res.edges[:-1]:
    if e['au'] >= 0.95:
        ax.plot([], [], ' ')  # legend placeholder
ax.set_title(f'cohort_A — {METHOD_DIST} / {METHOD_HCLUST}', fontweight='bold')
ax.set_ylabel('distance')
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=8)
plt.tight_layout(); plt.show()

print('Clusters at AU >= 0.95:')
for e in pvpick(res, 0.95):
    print(f"  AU={e['au']:.3f}  {', '.join(e['members'])}")

## k-means, with the same AU treatment

`msfit` never asks where a cluster came from — only whether it is present or
absent in each bootstrap replicate. A k-means cluster qualifies, so k-means gets
AU *p*-values from the identical machinery.

Naming clusters by their **member set** also disposes of k-means' usual nuisance:
cluster labels are arbitrarily permuted between runs, which stops mattering once
a cluster is identified by what is in it.

> **Note.** Exact member-set matching is faithful to the published framework. The
> `jaccard=` option relaxes it (common practice for k-means stability) but changes
> the estimand — it is then no longer the AU *p*-value as published. Say which you used.

In [ ]:
for k in [2, 3, 4]:
    km = kmeans_pv(df.to_numpy(float), list(df.columns), k=k, nboot=200, seed=42)
    print(f'k={k}')
    for e in sorted(km.edges, key=lambda e: -e['au']):
        print(f"   AU={e['au']:.3f}  BP={e['bp']:.3f}  n={e['n_members']:2d}")

### Which k is best supported?

Instead of an elbow plot, ask which k produces clusters that survive resampling.

In [ ]:
rows = []
for k in range(2, 7):
    km = kmeans_pv(df.to_numpy(float), list(df.columns), k=k, nboot=200, seed=42)
    aus = [e['au'] for e in km.edges]
    rows.append({'k': k, 'min AU': min(aus), 'median AU': float(np.median(aus)),
                 'clusters AU>=0.95': sum(a >= 0.95 for a in aus)})
pd.DataFrame(rows)

---
**Next:** `across_cohorts` pools these cohorts. Each ships only aggregate summaries —
no sample-level rows — and the combined result is compared against what each
cohort could see alone.